# Berqo Field Scout — train the offline apricot model

Choose **Runtime → Change runtime type → T4 GPU** if available. Run cells in order. No paid AI API is needed. Use AgriVision-Apricot **baseline/original** images only. Dataset: https://data.mendeley.com/datasets/5wk5wpzczw/1 (Bilal Jan, CC BY 4.0). The exported model is a research prototype, not a diagnosis.

If you already downloaded the dataset, extract `apricot_baseline_images`, ZIP that folder and upload the smaller ZIP to Google Drive as `apricot_baseline_images.zip`. You do not need the augmented images.

In [ ]:
!pip -q install torch torchvision onnx onnxruntime numpy Pillow
import torch
print("GPU available:", torch.cuda.is_available())


In [ ]:
%%writefile /content/prepare_dataset.py
"""Prepare original apricot images, deduplicate before splitting, preserve a held-out test set."""
import argparse
import hashlib
import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image, ImageOps

SUPPORTED = {'healthy', 'brown_rot', 'shot_hole_leaves', 'shot_hole_fruit', 'powdery_mildew',
             'dieback', 'gumosis_and_galls', 'other_disorder_e_g_water_stress', 'wilting', 'unknown'}
DEFAULT_CLASSES = 'healthy,brown_rot,shot_hole_leaves,shot_hole_fruit'

def normalize(value):
    normalized = re.sub(r'[^a-z0-9]+', '_', value.lower()).strip('_')
    return {'gummosis_and_galls': 'gumosis_and_galls'}.get(normalized, normalized)

def label_for(path, root, classes):
    for folder in reversed(path.relative_to(root).parts[:-1]):
        normalized = normalize(folder)
        if normalized in classes:
            return normalized
    return None

def fingerprint(path):
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert('RGB')
        if min(image.size) < 64:
            raise ValueError('Image smaller than 64 pixels')
        exact = hashlib.sha256(image.tobytes() + str(image.size).encode()).hexdigest()
        pixels = list(image.convert('L').resize((9, 8)).getdata())
        perceptual = sum((pixels[y * 9 + x] > pixels[y * 9 + x + 1]) << (y * 8 + x)
                         for y in range(8) for x in range(8))
        return exact, perceptual

def prepare(root, output, classes, min_count=30, seed=42):
    root, output = Path(root).resolve(), Path(output).resolve()
    if not root.is_dir():
        raise ValueError(f'Dataset directory does not exist: {root}')
    if not set(classes) <= SUPPORTED or len(set(classes)) != len(classes) or len(classes) < 2:
        raise ValueError('Use at least two distinct supported apricot classes')
    output.mkdir(parents=True, exist_ok=True)
    images = []
    exact_seen, perceptual_seen = {}, {}
    duplicates, rejected, augmented = 0, [], 0
    for path in sorted(root.rglob('*')):
        if path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}:
            continue
        if any('augment' in part.lower() for part in path.relative_to(root).parts):
            augmented += 1
            continue
        label = label_for(path, root, classes)
        if label is None:
            continue
        try:
            exact, phash = fingerprint(path)
            previous = exact_seen.get(exact) or perceptual_seen.get(phash)
            if previous:
                if previous != label:
                    raise ValueError(f'Conflicting duplicate labels: {previous} / {label}')
                duplicates += 1
                continue
            exact_seen[exact] = label
            perceptual_seen[phash] = label
            # Cache lossless resized originals once so CPU training does not decode multi-MB photos every epoch.
            cached = output / 'prepared' / label / f'{exact}.png'
            cached.parent.mkdir(parents=True, exist_ok=True)
            if not cached.exists():
                with Image.open(path) as source:
                    ImageOps.exif_transpose(source).convert('RGB').resize((224, 224), Image.Resampling.BILINEAR).save(cached)
            images.append({'path': str(cached), 'originalPath': str(path), 'label': label, 'sha256': exact, 'dhash': str(phash)})
        except Exception as error:
            rejected.append({'path': str(path), 'error': str(error)})
    if any('Conflicting duplicate' in item['error'] for item in rejected):
        raise ValueError('Duplicate images have conflicting labels. Correct labels before training.')
    grouped = defaultdict(list)
    for item in images:
        grouped[item['label']].append(item)
    rng = random.Random(seed)
    splits = {'train': [], 'val': [], 'test': []}
    for label in classes:
        items = grouped[label]
        if len(items) < min_count:
            raise ValueError(f'{label}: only {len(items)} unique original images; need {min_count}. Select fewer classes or fix --data path.')
        rng.shuffle(items)
        holdout = max(3, round(len(items) * .15))
        splits['test'].extend(items[:holdout])
        splits['val'].extend(items[holdout:2 * holdout])
        splits['train'].extend(items[2 * holdout:])
    manifest = {'classes': classes, 'seed': seed, 'root': str(root), 'splits': splits,
                'counts': {name: dict(Counter(i['label'] for i in values)) for name, values in splits.items()},
                'duplicatesRemoved': duplicates, 'augmentedFilesSkipped': augmented, 'rejected': rejected,
                'limitations': ['Exact and identical difference-hash duplicates removed. Similar scenes from the same tree may remain. Group by orchard/tree before production validation.',
                                'Random image split is not an independent Armenian orchard evaluation.']}
    (output / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(json.dumps({k: v for k, v in manifest.items() if k not in {'splits', 'rejected'}}, indent=2))
    return manifest

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', required=True)
    parser.add_argument('--output', default='ml/outputs')
    parser.add_argument('--classes', default=DEFAULT_CLASSES)
    parser.add_argument('--min-count', type=int, default=30)
    args = parser.parse_args()
    prepare(args.data, args.output, args.classes.split(','), args.min_count)


In [ ]:
%%writefile /content/train.py
"""MobileNetV3 Small transfer learning + held-out metrics + mobile-compatible ONNX export."""
import argparse
import json
import random
import time
import zipfile
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageOps
import onnx
import onnxruntime as ort

MEAN, STD = [.485, .456, .406], [.229, .224, .225]
SOURCE = {'title': 'AgriVision-Apricot — Bilal Jan, version 1',
          'url': 'https://data.mendeley.com/datasets/5wk5wpzczw/1', 'license': 'CC BY 4.0'}

class Images(Dataset):
    def __init__(self, items, classes, train=False):
        self.items, self.classes = items, classes
        steps = [transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR)]
        if train:
            steps += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(12),
                      transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.12)]
        self.transform = transforms.Compose(steps + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    def __len__(self): return len(self.items)
    def __getitem__(self, index):
        item = self.items[index]
        with Image.open(item['path']) as image:
            tensor = self.transform(ImageOps.exif_transpose(image).convert('RGB'))
        return tensor, self.classes.index(item['label'])

def score(model, loader, device):
    model.eval()
    predictions, targets, logits_all = [], [], []
    with torch.inference_mode():
        for inputs, labels in loader:
            logits = model(inputs.to(device)).cpu()
            predictions.extend(logits.argmax(1).tolist())
            targets.extend(labels.tolist())
            logits_all.append(logits)
    return np.array(predictions), np.array(targets), torch.cat(logits_all)

def metrics(predictions, targets, classes):
    confusion = np.zeros((len(classes), len(classes)), dtype=int)
    for actual, predicted in zip(targets, predictions): confusion[actual, predicted] += 1
    per_class = {}
    for i, name in enumerate(classes):
        tp = int(confusion[i, i]); precision = tp / max(1, int(confusion[:, i].sum()))
        recall = tp / max(1, int(confusion[i, :].sum()))
        per_class[name] = {'precision': precision, 'recall': recall,
                           'f1': 2 * precision * recall / max(1e-12, precision + recall),
                           'testCount': int(confusion[i, :].sum())}
    return {'accuracy': float((predictions == targets).mean()),
            'macroF1': float(np.mean([c['f1'] for c in per_class.values()])),
            'testCount': len(targets), 'perClass': per_class, 'confusionMatrix': confusion.tolist()}

def train(args):
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    torch.set_num_threads(min(8, max(1, torch.get_num_threads())))
    output = Path(args.output); output.mkdir(parents=True, exist_ok=True)
    manifest = json.loads(Path(args.manifest).read_text(encoding='utf-8'))
    classes = manifest['classes']
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}; classes: {classes}', flush=True)
    datasets = {split: Images(items, classes, train=split == 'train') for split, items in manifest['splits'].items()}
    loaders = {split: DataLoader(data, batch_size=args.batch_size, shuffle=split == 'train', num_workers=0) for split, data in datasets.items()}
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    for parameter in model.features.parameters(): parameter.requires_grad = False
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(classes))
    model.to(device)
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=.001, weight_decay=.01)
    counts = np.bincount([classes.index(i['label']) for i in manifest['splits']['train']], minlength=len(classes))
    weights = torch.tensor(counts.sum() / (len(classes) * counts), dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    best_f1 = -1
    start = time.time()
    for epoch in range(args.epochs):
        if epoch == args.epochs - args.fine_tune_epochs:
            for parameter in model.features[-3:].parameters(): parameter.requires_grad = True
            optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=.0001, weight_decay=.01)
        model.train(); model.features.eval()  # frozen batch-norm statistics on a small dataset
        running = 0.
        for inputs, labels in loaders['train']:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(inputs.to(device)), labels.to(device))
            loss.backward(); optimizer.step(); running += loss.item()
        predictions, targets, _ = score(model, loaders['val'], device)
        validation = metrics(predictions, targets, classes)
        print(f'Epoch {epoch + 1}/{args.epochs}: loss={running / len(loaders["train"]):.3f} val_accuracy={validation["accuracy"]:.3f} val_macro_f1={validation["macroF1"]:.3f}', flush=True)
        if validation['macroF1'] > best_f1:
            best_f1 = validation['macroF1']
            torch.save(model.state_dict(), output / 'best.pt')
    model.load_state_dict(torch.load(output / 'best.pt', map_location=device, weights_only=True))
    val_pred, val_targets, val_logits = score(model, loaders['val'], device)
    # Fit temperature on validation only. Test data is not used for tuning.
    temperatures = [.75, 1., 1.25, 1.5, 2., 2.5, 3.]
    temperature = min(temperatures, key=lambda t: nn.functional.cross_entropy(val_logits / t, torch.tensor(val_targets)).item())
    model = nn.Sequential(model, Temperature(temperature)).eval().to(device)
    probabilities = (val_logits / temperature).softmax(1)
    top_two = probabilities.topk(2, dim=1).values
    threshold, threshold_validated = 1., False
    for candidate in np.arange(.8, 1., .02):
        accepted = (top_two[:, 0] >= candidate) & ((top_two[:, 0] - top_two[:, 1]) >= .15)
        if 'unknown' in classes: accepted &= torch.tensor(val_pred != classes.index('unknown'))
        if int(accepted.sum()) >= 5 and float(torch.tensor(val_pred == val_targets)[accepted].float().mean()) >= .9:
            threshold, threshold_validated = float(candidate), True
            break
    predictions, targets, _ = score(model, loaders['test'], device)
    report = metrics(predictions, targets, classes)
    report.update({'classes': classes, 'validationMacroF1': best_f1, 'temperature': temperature,
                   'confidenceThreshold': threshold, 'thresholdValidationPassed': threshold_validated,
                   'trainingSeconds': time.time() - start, 'device': str(device), 'counts': manifest['counts']})
    model = model.cpu().eval()
    sample = torch.zeros(1, 3, 224, 224)
    torch.onnx.export(model, sample, str(output / 'model.onnx'), input_names=['image'], output_names=['logits'], opset_version=17, dynamo=False)
    onnx.checker.check_model(onnx.load(str(output / 'model.onnx')))
    session = ort.InferenceSession(str(output / 'model.onnx'), providers=['CPUExecutionProvider'])
    probe = datasets['test'][0][0].unsqueeze(0)
    with torch.inference_mode(): expected = model(probe).numpy()
    actual = session.run(['logits'], {'image': probe.numpy()})[0]
    if not np.allclose(expected, actual, atol=1e-4, rtol=1e-3): raise ValueError('ONNX / PyTorch parity failed')
    report['onnxParityMaxError'] = float(np.abs(expected - actual).max())
    report['modelBytes'] = (output / 'model.onnx').stat().st_size
    limitations = manifest['limitations'] + [
        'Research prototype: predictions are not diagnoses or treatment instructions.',
        'No independent validation on Armenian orchards. Similar symptoms and mixed problems may be confused.',
        'Confidence and margin do not reliably detect unsupported objects. Inspect only apricot.',
        'Only the listed classes are supported. Other apricot problems are not covered.',
        'Float32 model chosen for export reliability. Quantization deferred until accuracy is measured.',
        'Mobile JPEG resize can differ slightly from Pillow interpolation; real-device comparison is still required.']
    if not threshold_validated: limitations.append('Validation failed to establish >=90% selective precision with >=5 accepted samples. App rejects all results until more data/training is available.')
    metadata = {'modelId': 'berqo-apricot-mobilenetv3-small', 'version': args.version,
                'classes': classes, 'inputSize': 224, 'inputName': 'image', 'outputName': 'logits',
                'inputLayout': 'NCHW', 'outputType': 'logits', 'mean': MEAN, 'std': STD,
                'confidenceThreshold': threshold, 'marginThreshold': .15,
                'trainingSources': [SOURCE], 'limitations': limitations,
                'metrics': {k: report[k] for k in ['accuracy', 'macroF1', 'testCount']}}
    (output / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    (output / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    # Preserve genuinely held-out photos for the demo, without redistributing the whole dataset.
    with zipfile.ZipFile(output / 'apricot-crop-pack.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in ['model.onnx', 'metadata.json', 'metrics.json']: archive.write(output / name, name)
        for label in classes:
            items = [i for i in manifest['splits']['test'] if i['label'] == label][:2]
            for i, item in enumerate(items):
                original = item.get('originalPath', item['path'])
                archive.write(original, f'held-out-demo/{label}-{i}{Path(original).suffix}')
    print(json.dumps(report, indent=2), flush=True)
    print(f'Export complete: {output / "apricot-crop-pack.zip"}', flush=True)

class Temperature(nn.Module):
    def __init__(self, value): super().__init__(); self.value = value
    def forward(self, inputs): return inputs / self.value

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--manifest', default='ml/outputs/manifest.json')
    parser.add_argument('--output', default='ml/outputs')
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--fine-tune-epochs', type=int, default=3)
    parser.add_argument('--batch-size', type=int, default=64)
    parser.add_argument('--version', default='0.1.0')
    args = parser.parse_args()
    if args.epochs < 1 or not 0 <= args.fine_tune_epochs <= args.epochs: parser.error('Invalid epoch counts')
    train(args)


## Choose data input

Option A mounts your Drive. Set `DATA_ZIP` below to the uploaded archive. Option B downloads the official full archive directly into Colab (8.93 GB; may be blocked by the host). Choose one. Archive extraction selects only baseline images; it skips augmented images.

In [ ]:
USE_DRIVE = True
DATA_ZIP = "/content/drive/MyDrive/apricot_baseline_images.zip"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    import urllib.request
    DATA_ZIP = "/content/apricot.zip"
    urllib.request.urlretrieve("https://data.mendeley.com/public-api/zip/5wk5wpzczw/download/1", DATA_ZIP)


In [ ]:
from pathlib import Path
import zipfile, shutil
DATA_ROOT = Path("/content/apricot")
DATA_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as archive:
    members = [m for m in archive.infolist() if not m.is_dir() and not any("augment" in p.lower() for p in Path(m.filename).parts)]
    total = sum(m.file_size for m in members)
    print("Original files:", len(members), "uncompressed GB:", round(total/1e9,2))
    if total > 20_000_000_000: raise ValueError("Archive unexpectedly large")
    for member in members:
        relative = Path(member.filename)
        if relative.is_absolute() or ".." in relative.parts: raise ValueError("Unsafe archive path")
        destination = DATA_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
print("Extracted original images into", DATA_ROOT)


In [ ]:
!python /content/prepare_dataset.py --data /content/apricot --output /content/outputs --classes healthy,brown_rot,shot_hole_leaves,shot_hole_fruit


## Train, evaluate and export

Start with 10 epochs. Validation chooses the checkpoint and confidence threshold. The test split stays held out. If selective validation does not pass, the exported pack rejects results instead of pretending reliability. Do not lower the threshold just to make a demo look better. Field photos from Armenian orchards are needed for production validation.

In [ ]:
!python /content/train.py --manifest /content/outputs/manifest.json --output /content/outputs --epochs 10 --batch-size 64


In [ ]:
import json
from pathlib import Path
print(Path("/content/outputs/metrics.json").read_text())
from google.colab import files
files.download("/content/outputs/apricot-crop-pack.zip")


## Put the model in the app

Extract `apricot-crop-pack.zip` into `D:\Projects\BerqoHackathon\ml\outputs`. In the project terminal run `npm run bundle:model`, then `npm run build:android`. The resulting APK contains the model and starts offline. Alternatively, transfer `model.onnx` and `metadata.json` to the phone and install them in the APK Settings screen. Expo Go cannot run the native ONNX runtime.

Keep `metrics.json` for judging. Use the ZIP's `held-out-demo` images on the phone. Do not claim Armenian orchard accuracy from this dataset evaluation.